# Hybrid Residual V2

V1 chỉ dùng ARX backbone `(1,5,2)`. V2 search nhiều backbone ARX trọng điểm rồi train residual correction cho từng backbone:

- Backbone lấy từ top ARX order search.
- Residual models: Ridge, HGBR, ExtraTrees, RandomForest.
- Chọn final bằng validation `FIT_sim`.


In [1]:
from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

WORK_DIR = Path.cwd()
PROJECT_ROOT = WORK_DIR.parent if WORK_DIR.name == "Hybrid_ARX_NARX" else WORK_DIR
OUT_DIR = PROJECT_ROOT / "Hybrid_ARX_NARX"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from arx_pipeline import (
    DataConfig,
    SplitConfig,
    ModelConfig,
    load_or_generate_data,
    split_time_series,
    build_regression_matrix,
    estimate_ols,
    simulate_arx,
    compute_metrics,
)
from narx_pipeline import (
    build_augmented_df,
    fit_zscore_stats,
    apply_zscore,
    inverse_zscore_y,
    scaled_clip_bounds,
)

BASELINE_INPUT_COLS = ("Temperature", "Humidity", "Light", "Drip", "Mist", "Fan")
AUGMENTED_INPUT_COLS = (
    *BASELINE_INPUT_COLS,
    "Light_log",
    "Temp_x_Humi",
    "Temp_x_Light",
    "Humi_x_Light",
    "SP_Center",
    "SP_Width",
    "Month_sin",
    "Month_cos",
    "Season_sin",
    "Season_cos",
)
SCALE_COLS = ("Soil_Moisture", *AUGMENTED_INPUT_COLS)
SHRINK_CANDIDATES = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

BACKBONE_ORDERS = [
    (1, 5, 2),
    (5, 1, 2),
    (2, 5, 2),
    (5, 3, 2),
    (1, 1, 2),
    (1, 3, 2),
    (3, 1, 2),
    (2, 1, 2),
]


## 1. Data


In [2]:
df_full, _, data_source = load_or_generate_data(
    DataConfig(
        csv_path=PROJECT_ROOT / "greenhouse_data.csv",
        generator_script_path=PROJECT_ROOT / "data_generator.py",
        force_regenerate_from_script=False,
        auto_save_generated_csv=True,
    )
)
df_aug = build_augmented_df(df_full)
df_train, df_val, df_test = split_time_series(df_aug, SplitConfig(train_ratio=0.60, val_ratio=0.20))

scale_stats = fit_zscore_stats(df_train, SCALE_COLS)
clip_bounds_real, clip_bounds_scaled = scaled_clip_bounds(df_train, scale_stats, (0.01, 0.99))
df_train_z = apply_zscore(df_train, scale_stats)
df_val_z = apply_zscore(df_val, scale_stats)
df_test_z = apply_zscore(df_test, scale_stats)


## 2. Helpers


In [3]:
def build_residual_features(df_z: pd.DataFrame, y_arx_sim: np.ndarray, cfg: ModelConfig) -> np.ndarray:
    lag = max(cfg.na, cfg.nb + cfg.nk - 1)
    y_arx_full = df_z[cfg.output_col].astype(float).to_numpy().copy()
    y_arx_full[lag:] = y_arx_sim
    rows = []
    for idx, t in enumerate(range(lag, len(df_z))):
        row = [float(y_arx_sim[idx])]
        for y_lag in range(1, 4):
            if t - y_lag >= 0:
                row.append(float(y_arx_full[t - y_lag]))
            else:
                row.append(float(y_arx_full[0]))
        for col in AUGMENTED_INPUT_COLS:
            values = df_z[col].astype(float).to_numpy()
            row.append(float(values[t]))
            row.append(float(values[max(0, t - 1)]))
            row.append(float(values[max(0, t - 2)]))
        rows.append(row)
    return np.asarray(rows, dtype=float)


def metric_real(y_true_z: np.ndarray, y_pred_z: np.ndarray, n_params: int = 0) -> dict[str, float]:
    return compute_metrics(
        inverse_zscore_y(y_true_z, scale_stats),
        inverse_zscore_y(y_pred_z, scale_stats),
        n_params,
    )


def evaluate_hybrid(y_true_z: np.ndarray, y_arx_z: np.ndarray, correction_z: np.ndarray, shrink: float) -> dict[str, float]:
    y_hybrid_z = y_arx_z + shrink * correction_z
    y_hybrid_z = np.clip(y_hybrid_z, clip_bounds_scaled[0], clip_bounds_scaled[1])
    return metric_real(y_true_z, y_hybrid_z)


def residual_models(seed: int):
    return [
        ("ridge_1", make_pipeline(StandardScaler(), Ridge(alpha=1.0))),
        ("hgb_31", HistGradientBoostingRegressor(
            max_iter=180,
            learning_rate=0.05,
            max_leaf_nodes=31,
            l2_regularization=0.0,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=15,
            random_state=seed,
        )),
        ("extra_trees_leaf5", ExtraTreesRegressor(
            n_estimators=120,
            max_features=0.7,
            min_samples_leaf=5,
            n_jobs=-1,
            random_state=seed + 1,
        )),
        ("random_forest_leaf5", RandomForestRegressor(
            n_estimators=120,
            max_features=0.7,
            min_samples_leaf=5,
            n_jobs=-1,
            random_state=seed + 2,
        )),
    ]


## 3. Backbone + residual search


In [4]:
rows = []
backbone_rows = []
candidate_id = 0

for backbone_idx, (na, nb, nk) in enumerate(BACKBONE_ORDERS, start=1):
    cfg = ModelConfig(
        na=na,
        nb=nb,
        nk=nk,
        include_intercept=True,
        input_cols=AUGMENTED_INPUT_COLS,
        simulation_clip=clip_bounds_scaled,
    )
    x_arx_train, y_arx_train_target = build_regression_matrix(df_train_z, cfg)
    theta_arx, _, sigma2_arx = estimate_ols(x_arx_train, y_arx_train_target)

    y_arx_train, y_true_train = simulate_arx(df_train_z, theta_arx, cfg)
    y_arx_val, y_true_val = simulate_arx(df_val_z, theta_arx, cfg)
    y_arx_test, y_true_test = simulate_arx(df_test_z, theta_arx, cfg)

    arx_val = metric_real(y_true_val, y_arx_val, x_arx_train.shape[1])
    arx_test = metric_real(y_true_test, y_arx_test, x_arx_train.shape[1])
    order_label = f"({na},{nb},{nk})"
    backbone_rows.append({
        "order": order_label,
        "na": na,
        "nb": nb,
        "nk": nk,
        "arx_val_FIT_sim": arx_val["FIT"],
        "arx_test_FIT_sim": arx_test["FIT"],
        "arx_test_RMSE_sim": arx_test["RMSE"],
    })

    x_res_train = build_residual_features(df_train_z, y_arx_train, cfg)
    x_res_val = build_residual_features(df_val_z, y_arx_val, cfg)
    x_res_test = build_residual_features(df_test_z, y_arx_test, cfg)
    y_res_train = y_true_train - y_arx_train

    for model_name, model in residual_models(seed=100 + backbone_idx * 10):
        candidate_id += 1
        start = time.time()
        model.fit(x_res_train, y_res_train)
        train_seconds = time.time() - start
        corr_val = model.predict(x_res_val)
        corr_test = model.predict(x_res_test)

        for shrink in SHRINK_CANDIDATES:
            val_hybrid = evaluate_hybrid(y_true_val, y_arx_val, corr_val, shrink)
            test_hybrid = evaluate_hybrid(y_true_test, y_arx_test, corr_test, shrink)
            rows.append({
                "candidate_id": candidate_id,
                "order": order_label,
                "na": na,
                "nb": nb,
                "nk": nk,
                "residual_model": model_name,
                "shrink": shrink,
                "train_seconds": train_seconds,
                "arx_val_FIT_sim": arx_val["FIT"],
                "arx_test_FIT_sim": arx_test["FIT"],
                "val_FIT_sim": val_hybrid["FIT"],
                "val_RMSE_sim": val_hybrid["RMSE"],
                "test_FIT_sim": test_hybrid["FIT"],
                "test_RMSE_sim": test_hybrid["RMSE"],
                "test_Bias_sim": test_hybrid["Bias"],
            })
    print(f"done backbone {order_label}")

backbone_df = pd.DataFrame(backbone_rows).sort_values("arx_val_FIT_sim", ascending=False).reset_index(drop=True)
search_df = pd.DataFrame(rows).sort_values(["val_FIT_sim", "test_FIT_sim"], ascending=[False, False]).reset_index(drop=True)
best_by_validation = search_df.iloc[0].to_dict()

display(backbone_df.round(4))
search_df.head(25).round(4)


done backbone (1,5,2)


done backbone (5,1,2)


done backbone (2,5,2)


done backbone (5,3,2)


done backbone (1,1,2)


done backbone (1,3,2)


done backbone (3,1,2)


done backbone (2,1,2)


,order,na,nb,nk,arx_val_FIT_sim,arx_test_FIT_sim,arx_test_RMSE_sim
0,"(1,5,2)",1,5,2,68.9559,66.8337,0.9661
1,"(5,1,2)",5,1,2,68.8411,66.4176,0.9782
2,"(2,5,2)",2,5,2,68.7952,66.4020,0.9787
3,"(5,3,2)",5,3,2,68.7876,66.4922,0.9760
4,"(1,1,2)",1,1,2,68.2521,66.2635,0.9826
5,"(1,3,2)",1,3,2,68.2398,66.1694,0.9854
6,"(3,1,2)",3,1,2,68.1768,65.8784,0.9939
7,"(2,1,2)",2,1,2,68.1595,65.9517,0.9917


,candidate_id,order,na,nb,nk,residual_model,shrink,train_seconds,arx_val_FIT_sim,arx_test_FIT_sim,val_FIT_sim,val_RMSE_sim,test_FIT_sim,test_RMSE_sim,test_Bias_sim
0,16,"(5,3,2)",5,3,2,random_forest_leaf5,0.8,21.3819,68.7876,66.4922,71.3271,0.8608,69.3966,0.8914,-0.0063
1,16,"(5,3,2)",5,3,2,random_forest_leaf5,1.0,21.3819,68.7876,66.4922,71.2904,0.8619,69.3776,0.8920,-0.0117
2,4,"(1,5,2)",1,5,2,random_forest_leaf5,0.8,19.9042,68.9559,66.8337,71.2555,0.8630,69.4110,0.8910,0.0011
3,12,"(2,5,2)",2,5,2,random_forest_leaf5,0.8,19.5806,68.7952,66.4020,71.2507,0.8632,69.2459,0.8958,0.0030
4,26,"(3,1,2)",3,1,2,hgb_31,1.0,1.1917,68.1768,65.8784,71.2278,0.8638,69.5140,0.8880,0.0016
5,12,"(2,5,2)",2,5,2,random_forest_leaf5,1.0,19.5806,68.7952,66.4020,71.2186,0.8641,69.2499,0.8957,-0.0024
6,4,"(1,5,2)",1,5,2,random_forest_leaf5,1.0,19.9042,68.9559,66.8337,71.2022,0.8646,69.3676,0.8923,-0.0044
7,24,"(1,3,2)",1,3,2,random_forest_leaf5,0.8,20.5106,68.2398,66.1694,71.2019,0.8646,69.5027,0.8883,0.0072
8,24,"(1,3,2)",1,3,2,random_forest_leaf5,1.0,20.5106,68.2398,66.1694,71.1911,0.8649,69.4828,0.8889,0.0015
9,30,"(2,1,2)",2,1,2,hgb_31,1.0,1.1355,68.1595,65.9517,71.1890,0.8649,69.5092,0.8881,-0.0281


## 4. Comparison


In [5]:
comparison_rows = []

arx_best_path = PROJECT_ROOT / "ARX_Model_VersionSearch" / "arx_order_search_v1.json"
if arx_best_path.exists():
    with arx_best_path.open("r", encoding="utf-8") as f:
        arx_search = json.load(f)["best_by_validation"]
    arx_best_test = arx_search["test_FIT_sim"]
    comparison_rows.append({
        "model": "ARX Search V1 best",
        "val_FIT_sim": arx_search["val_FIT_sim"],
        "test_FIT_sim": arx_search["test_FIT_sim"],
        "test_RMSE_sim": arx_search["test_RMSE_sim"],
        "test_gain_vs_arx": 0.0,
    })
else:
    arx_best_test = float(backbone_df.iloc[0]["arx_test_FIT_sim"])

v1_path = OUT_DIR / "hybrid_residual_v1.json"
if v1_path.exists():
    with v1_path.open("r", encoding="utf-8") as f:
        v1 = json.load(f)["best_by_validation"]
    comparison_rows.append({
        "model": "Hybrid Residual V1",
        "val_FIT_sim": v1["val_FIT_sim"],
        "test_FIT_sim": v1["test_FIT_sim"],
        "test_RMSE_sim": v1["test_RMSE_sim"],
        "test_gain_vs_arx": v1["test_FIT_sim"] - arx_best_test,
    })

comparison_rows.append({
    "model": "Hybrid Residual V2 best-by-val",
    "val_FIT_sim": best_by_validation["val_FIT_sim"],
    "test_FIT_sim": best_by_validation["test_FIT_sim"],
    "test_RMSE_sim": best_by_validation["test_RMSE_sim"],
    "test_gain_vs_arx": best_by_validation["test_FIT_sim"] - arx_best_test,
})

narx_v6_path = PROJECT_ROOT / "NARX" / "narx_v6.json"
if narx_v6_path.exists():
    with narx_v6_path.open("r", encoding="utf-8") as f:
        narx_v6 = json.load(f)["selected_candidate"]
    comparison_rows.append({
        "model": "NARX V6 selected",
        "val_FIT_sim": narx_v6["val_FIT_sim"],
        "test_FIT_sim": narx_v6["test_FIT_sim"],
        "test_RMSE_sim": narx_v6["test_RMSE_sim"],
        "test_gain_vs_arx": narx_v6["test_FIT_sim"] - arx_best_test,
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values("test_FIT_sim", ascending=False).reset_index(drop=True)
comparison_df.round(4)


,model,val_FIT_sim,test_FIT_sim,test_RMSE_sim,test_gain_vs_arx
0,Hybrid Residual V1,71.2298,69.4550,0.8898,2.6213
1,Hybrid Residual V2 best-by-val,71.3271,69.3966,0.8914,2.5628
2,NARX V6 selected,70.1837,68.5310,0.9166,1.6973
3,ARX Search V1 best,68.9559,66.8337,0.9661,0.0000


## 5. Save


In [6]:
def json_ready(value):
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    if isinstance(value, np.ndarray):
        return json_ready(value.tolist())
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, Path):
        return str(value)
    return value


def df_to_markdown(df: pd.DataFrame) -> str:
    df_str = df.astype(str)
    headers = list(df_str.columns)
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    for _, row in df_str.iterrows():
        lines.append("| " + " | ".join(str(row[col]) for col in headers) + " |")
    return "\n".join(lines) + "\n"


artifact = {
    "model_type": "Hybrid_ARX_Residual_NARX",
    "version": "hybrid_residual_v2_backbone_search",
    "backbone_orders": [list(v) for v in BACKBONE_ORDERS],
    "selection_metric": "validation FIT_sim",
    "best_by_validation": best_by_validation,
    "backbone_results": backbone_df.to_dict(orient="records"),
    "search_results": search_df.to_dict(orient="records"),
    "comparison": comparison_df.to_dict(orient="records"),
}

OUT_DIR.mkdir(exist_ok=True)
json_path = OUT_DIR / "hybrid_residual_v2.json"
search_csv_path = OUT_DIR / "hybrid_residual_v2_search.csv"
comparison_csv_path = OUT_DIR / "hybrid_residual_v2_comparison.csv"
comparison_md_path = OUT_DIR / "hybrid_residual_v2_comparison.md"
backbone_csv_path = OUT_DIR / "hybrid_residual_v2_backbones.csv"

with json_path.open("w", encoding="utf-8") as f:
    json.dump(json_ready(artifact), f, indent=2)
    f.write("\n")

search_df.to_csv(search_csv_path, index=False)
backbone_df.to_csv(backbone_csv_path, index=False)
comparison_df.to_csv(comparison_csv_path, index=False)
comparison_md_path.write_text(df_to_markdown(comparison_df.round(4)), encoding="utf-8")

json_path, search_csv_path, comparison_md_path


(WindowsPath('C:/Users/minht/OneDrive/Desktop/ARX-Model/Hybrid_ARX_NARX/hybrid_residual_v2.json'),
 WindowsPath('C:/Users/minht/OneDrive/Desktop/ARX-Model/Hybrid_ARX_NARX/hybrid_residual_v2_search.csv'),
 WindowsPath('C:/Users/minht/OneDrive/Desktop/ARX-Model/Hybrid_ARX_NARX/hybrid_residual_v2_comparison.md'))